# Multi-Provider LLM Client — Demo

This notebook demonstrates the `llm_client` module:
1. Same prompt sent to all 3 providers with a comparative table.
2. Structured output with a Pydantic schema.
3. Streaming with one provider.

> **Prerequisites**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GEMINI_API_KEY` in `../../.env` (repo root).

In [1]:
import sys
from pathlib import Path

# Ensure the package is importable when running from 04_multi_provider/
sys.path.insert(0, str(Path.cwd()))

from llm_client import (
    AnthropicProvider,
    GeminiProvider,
    LLMResponse,
    OpenAIProvider,
    StreamingResponse,
)

## 1 — Same prompt, all 3 providers

In [2]:
PROMPT = "Explain what a neural network is in 2 sentences."

openai = OpenAIProvider()
anthropic = AnthropicProvider()
gemini = GeminiProvider()

results: list[LLMResponse] = []

r_openai = await openai.generate(prompt=PROMPT, model="gpt-5.4")
r_anthropic = await anthropic.generate(prompt=PROMPT, model="claude-sonnet-4-6")
r_gemini = await gemini.generate(prompt=PROMPT, model="gemini-3.1-pro")

results = [r_openai, r_anthropic, r_gemini]
print("Calls complete.")

Calls complete.


In [3]:
import pandas as pd

rows = [
    {
        "provider": r.provider,
        "model": r.model,
        "response_text": r.text[:120] + "..." if len(r.text) > 120 else r.text,
        "input_tokens": r.input_tokens,
        "output_tokens": r.output_tokens,
        "cost_usd": f"${r.cost_usd:.6f}",
        "latency_ms": r.latency_ms,
    }
    for r in results
]

df = pd.DataFrame(rows)
df

,provider,model,response_text,input_tokens,output_tokens,cost_usd,latency_ms
0,openai,gpt-5.4,A neural network is a type of machine learning...,17,61,$0.000958,3450
1,anthropic,claude-sonnet-4-6,A neural network is a computational system ins...,20,63,$0.001005,1706
2,gemini,gemini-3.1-pro-preview,A neural network is a type of artificial intel...,12,50,$0.000624,9000


## 2 — Structured output with a Pydantic schema

In [4]:
from pydantic import BaseModel


class NeuralNetworkSummary(BaseModel):
    """Structured summary of the neural network explanation."""

    definition: str
    analogy: str
    use_case: str


STRUCTURED_PROMPT = (
    "Explain what a neural network is. "
    "Provide: a one-sentence definition, an analogy, and a real-world use case."
)

structured_result = await anthropic.generate(
    prompt=STRUCTURED_PROMPT,
    model="claude-sonnet-4-6",
    schema=NeuralNetworkSummary,
)

print("Raw text:", structured_result.text[:200])
print()
print("Parsed object:", structured_result.parsed)
print()
if structured_result.parsed:
    s: NeuralNetworkSummary = structured_result.parsed
    print(f"Definition : {s.definition}")
    print(f"Analogy    : {s.analogy}")
    print(f"Use case   : {s.use_case}")

Raw text: {"definition":"A neural network is a computational model inspired by the human brain, consisting of interconnected layers of nodes (neurons) that process and learn patterns from data to make predictio

Parsed object: definition='A neural network is a computational model inspired by the human brain, consisting of interconnected layers of nodes (neurons) that process and learn patterns from data to make predictions or decisions.' analogy='A neural network is like a team of specialists in an assembly line — each worker (neuron) handles a small piece of the task and passes their result to the next, and through repeated practice (training), the whole team learns to produce the correct final output with increasing accuracy.' use_case='Neural networks are used in medical imaging to detect diseases such as cancer, where the model is trained on thousands of labeled X-rays or MRI scans to accurately identify tumors that might be missed by the human eye.'

Definition : A neural network 

## 3 — Streaming with OpenAI

In [5]:
import sys

stream_result = await openai.generate(
    prompt=PROMPT,
    model="gpt-5.4",
    stream=True,
)

assert isinstance(stream_result, StreamingResponse)

print("Streaming response (token by token):")
print("-" * 50)

async for chunk in stream_result:
    print(chunk, end="", flush=True)

print()
print("-" * 50)

final = stream_result.final_response
print(f"\nFinal response assembled: {len(final.text)} chars")

Streaming response (token by token):
--------------------------------------------------
A neural network is a type of machine learning model inspired by the brain, made of layers of connected nodes that learn patterns from data. It works by adjusting the strength of those connections during training so it can make predictions or decisions, like recognizing images or understanding text.
--------------------------------------------------

Final response assembled: 300 chars
